# Nutrition5k mass estimation: pseudo-depth v3 try2

Adds plate segmentation features


In [1]:
from pathlib import Path
import json
import math
import os
import sys
import time
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor, Pool
from ultralytics import YOLO


c:\Projects\FoodProject\FoodProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Project paths and settings

In [2]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if all((path / name).exists() for name in ("DepthModule", "mass_estimation", "nutrition5k", "models")):
            return path
    raise RuntimeError("Project root was not found. Run the notebook from inside the cloned repository.")


def env_path(name: str, default: Path) -> Path:
    return Path(os.getenv(name, default)).expanduser()


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "DepthModule"))
from depth_module import DepthAnything3Module

MASS_DIR = PROJECT_ROOT / "mass_estimation"
RESULTS_DIR = MASS_DIR / "pseudo-depth-v3-try2"
MAPPING_DIR = PROJECT_ROOT / "class_mappings"

NUTRITION_CSV = env_path("NUTRITION_CSV", PROJECT_ROOT / "nutrition5k" / "dish_nutrition_values.csv")
OVERHEAD_DIR = env_path("OVERHEAD_DIR", PROJECT_ROOT / "nutrition5k" / "imagery" / "realsense_overhead")
SEG_MODEL_PATH = env_path("SEG_MODEL_PATH", PROJECT_ROOT / "models" / "yolo_food_seg.pt")
PLATE_MODEL_PATH = env_path("PLATE_MODEL_PATH", PROJECT_ROOT / "models" / "plate_seg.pt")

_cls_default = PROJECT_ROOT / "models" / "yolo_cls.pt"
_cls_override = os.getenv("CLS_MODEL_PATH")
CLS_MODEL_PATH = Path(_cls_override).expanduser() if _cls_override else (_cls_default if _cls_default.exists() else None)
DEPTH_MODEL_ID = os.getenv("DEPTH_MODEL", "depth-anything-v3-base")

FEATURES_CSV = RESULTS_DIR / "catboost_pseudodepth_v3_try2_features.csv"
MODEL_PATH = RESULTS_DIR / "catboost_pseudodepth_v3_try2.cbm"
METRICS_PATH = RESULTS_DIR / "catboost_pseudodepth_v3_try2_metrics.json"
VALID_REPORT_CSV = RESULTS_DIR / "catboost_pseudodepth_v3_try2_valid_report.csv"

RANDOM_STATE = 42
MAX_DISHES = None
RESUME_FEATURE_CACHE = True
YOLO_CONF = 0.25
YOLO_IMGSZ = 640
CACHE_EVERY_N_ROWS = 25

CATBOOST_PARAMS = dict(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8,
    random_seed=RANDOM_STATE,
    od_type="Iter",
    od_wait=150,
    verbose=100,
)

required_paths = [NUTRITION_CSV, OVERHEAD_DIR, SEG_MODEL_PATH, PLATE_MODEL_PATH, MAPPING_DIR]
missing = [str(p.relative_to(PROJECT_ROOT) if p.is_relative_to(PROJECT_ROOT) else p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required project paths: " + ", ".join(missing))

print("Project root:", PROJECT_ROOT)
print("Nutrition CSV:", NUTRITION_CSV.relative_to(PROJECT_ROOT))
print("Overhead images:", OVERHEAD_DIR.relative_to(PROJECT_ROOT))
print("Food segmentation model:", SEG_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Plate segmentation model:", PLATE_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Classification model:", CLS_MODEL_PATH.relative_to(PROJECT_ROOT) if CLS_MODEL_PATH else "disabled")
print("Depth model:", DEPTH_MODEL_ID)
print("Outputs:", RESULTS_DIR.relative_to(PROJECT_ROOT))


Project root: c:\Projects\FoodProject\FoodProject
Nutrition CSV: nutrition5k\dish_nutrition_values.csv
Overhead images: nutrition5k\imagery\realsense_overhead
Food segmentation model: models\yolo_food_seg.pt
Plate segmentation model: models\plate_seg.pt
Classification model: models\yolo_cls.pt
Depth model: depth-anything-v3-base
Outputs: mass_estimation\pseudo-depth-v3-try2


## 3. Metadata


In [3]:
seg_map = pd.read_csv(MAPPING_DIR / "foodseg103_density_groups.csv")
seg_by_id = seg_map.set_index("class_id").to_dict("index")
food101_map = pd.read_csv(MAPPING_DIR / "food101_dish_groups.csv")
food101_group_by_name = dict(zip(food101_map["class_name"].astype(str), food101_map["dish_group"].astype(str)))

density_groups = sorted(seg_map.loc[seg_map["use_for_mask"].astype(str).str.lower() == "true", "density_group"].unique())
nutrition_cols = ["calories", "fat", "carb", "protein"]
labels = (
    pd.read_csv(NUTRITION_CSV)[["dish_id", "mass", *nutrition_cols]]
    .dropna(subset=["dish_id", "mass"])
    .assign(rgb_path=lambda df: df["dish_id"].astype(str).map(lambda dish_id: OVERHEAD_DIR / dish_id / "rgb.png"))
)
labels = labels[labels["rgb_path"].map(Path.exists)].copy()
if MAX_DISHES is not None:
    labels = labels.head(MAX_DISHES).copy()

print("Density groups:", density_groups)
print("Food101 groups:", sorted(food101_map["dish_group"].unique()))
print(f"Rows with rgb.png and mass target: {len(labels):,}")
labels.head()


Density groups: ['bread', 'dairy_dessert', 'dairy_fat', 'dessert', 'fruit', 'fruit_dried', 'fruit_fat', 'leafy_veg', 'liquid', 'meat', 'meat_processed', 'mixed_main', 'mushroom', 'nuts', 'protein', 'sauce', 'seafood', 'seaweed', 'starch', 'starch_legume', 'starch_main', 'starch_veg', 'unknown', 'vegetable']
Food101 groups: ['bread', 'breakfast', 'dairy_dessert', 'dairy_fat', 'dessert', 'meat_main', 'mixed_main', 'protein', 'salad', 'sandwich', 'sauce', 'seafood', 'soup_liquid', 'starch', 'starch_legume', 'starch_main']
Rows with rgb.png and mass target: 3,244


,dish_id,mass,calories,fat,carb,protein,rgb_path
0,dish_1561662216,193.0,300.794281,12.387489,28.218290,18.633970,c:\Projects\FoodProject\FoodProject\nutrition5...
2,dish_1561662054,292.0,419.438782,23.838249,26.351543,25.910593,c:\Projects\FoodProject\FoodProject\nutrition5...
3,dish_1562008979,290.0,382.936646,22.224644,10.173570,35.345387,c:\Projects\FoodProject\FoodProject\nutrition5...
4,dish_1560455030,103.0,20.590000,0.148000,4.625000,0.956000,c:\Projects\FoodProject\FoodProject\nutrition5...
5,dish_1558372433,143.0,74.360001,0.286000,0.429000,20.020000,c:\Projects\FoodProject\FoodProject\nutrition5...


## 4. Feature engineering helpers


In [4]:
def resize_nearest(mask: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    if mask.shape[:2] == shape_hw:
        return mask.astype(bool)
    return cv2.resize(mask.astype(np.uint8), (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_NEAREST).astype(bool)

def resize_depth(depth: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    depth = np.asarray(depth, dtype=np.float32)
    if depth.shape[:2] == shape_hw:
        return depth
    return cv2.resize(depth, (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_LINEAR)

def normalize_depth(depth: np.ndarray) -> Dict[str, np.ndarray]:
    depth = np.asarray(depth, dtype=np.float32)
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        z = np.zeros_like(depth, dtype=np.float32)
        return {"raw": z, "p05p95": z, "iqr_z": z}
    p05, p25, p50, p75, p95 = np.percentile(finite, [5, 25, 50, 75, 95])
    scale = max(float(p95 - p05), 1e-6)
    iqr = max(float(p75 - p25), 1e-6)
    p05p95 = np.clip((depth - p05) / scale, 0.0, 1.0).astype(np.float32)
    iqr_z = np.clip((depth - p50) / iqr, -5.0, 5.0).astype(np.float32)
    return {"raw": depth, "p05p95": p05p95, "iqr_z": iqr_z}

def mask_shape_features(mask: np.ndarray, prefix: str) -> Dict[str, float]:
    mask_u8 = mask.astype(np.uint8)
    area = int(mask_u8.sum())
    out = {f"{prefix}_area_px": area}
    if area == 0:
        out.update({
            f"{prefix}_perimeter": 0.0,
            f"{prefix}_compactness": 0.0,
            f"{prefix}_solidity": 0.0,
            f"{prefix}_equiv_diameter": 0.0,
        })
        return out
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = float(sum(cv2.arcLength(c, True) for c in contours))
    hull_area = 0.0
    for c in contours:
        if len(c) >= 3:
            hull_area += float(cv2.contourArea(cv2.convexHull(c)))
    out.update({
        f"{prefix}_perimeter": perimeter,
        f"{prefix}_compactness": float(area / max(perimeter * perimeter, 1e-6)),
        f"{prefix}_solidity": float(area / max(hull_area, 1e-6)),
        f"{prefix}_equiv_diameter": float(math.sqrt(4.0 * area / math.pi)),
    })
    return out


In [5]:
def predict_seg_instances(seg_model: YOLO, image: Image.Image) -> Tuple[List[Dict[str, object]], Dict[str, float]]:
    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    result = seg_model.predict(image_np, retina_masks=True, conf=YOLO_CONF, imgsz=YOLO_IMGSZ, verbose=False)[0]
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return [], {"n_masks_raw": 0, "n_masks_kept": 0, "seg_conf_mean": 0.0, "seg_conf_max": 0.0}

    raw_masks = result.masks.data.cpu().numpy() > 0.5
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy().astype(float)

    instances = []
    for raw_mask, class_id, conf in zip(raw_masks, classes, confs):
        meta = seg_by_id.get(int(class_id), {})
        use_for_mask = str(meta.get("use_for_mask", "true")).lower() == "true"
        if not use_for_mask:
            continue
        mask = resize_nearest(raw_mask, (h, w))
        area = int(mask.sum())
        if area == 0:
            continue
        instances.append({
            "mask": mask,
            "class_id": int(class_id),
            "class_name": str(meta.get("class_name", seg_model.names.get(int(class_id), class_id))),
            "density_group": str(meta.get("density_group", "unknown")),
            "conf": float(conf),
            "area": area,
        })

    stats = {
        "n_masks_raw": int(len(raw_masks)),
        "n_masks_kept": int(len(instances)),
        "seg_conf_mean": float(np.mean(confs)) if len(confs) else 0.0,
        "seg_conf_max": float(np.max(confs)) if len(confs) else 0.0,
    }
    return instances, stats


def predict_plate_mask(plate_model: YOLO, image: Image.Image) -> Tuple[np.ndarray, Dict[str, float]]:
    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    result = plate_model.predict(image_np, retina_masks=True, conf=YOLO_CONF, imgsz=YOLO_IMGSZ, verbose=False)[0]
    empty = np.zeros((h, w), dtype=bool)
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return empty, {"n_plate_masks": 0, "plate_conf_mean": 0.0, "plate_conf_max": 0.0}

    raw_masks = result.masks.data.cpu().numpy() > 0.5
    confs = result.boxes.conf.cpu().numpy().astype(float)
    masks = [resize_nearest(raw_mask, (h, w)) for raw_mask in raw_masks]
    masks = [mask for mask in masks if int(mask.sum()) > 0]
    if not masks:
        return empty, {"n_plate_masks": int(len(raw_masks)), "plate_conf_mean": float(np.mean(confs)), "plate_conf_max": float(np.max(confs))}

    plate_mask = np.logical_or.reduce(masks)
    stats = {
        "n_plate_masks": int(len(raw_masks)),
        "plate_conf_mean": float(np.mean(confs)) if len(confs) else 0.0,
        "plate_conf_max": float(np.max(confs)) if len(confs) else 0.0,
    }
    return plate_mask, stats


def predict_food101(cls_model: Optional[YOLO], image: Image.Image, topk: int = 5) -> Dict[str, object]:
    out = {"food101_top1": "unknown", "food101_top1_conf": 0.0, "food101_dish_group": "unknown", "food101_entropy": 0.0}
    for k in range(2, topk + 1):
        out[f"food101_top{k}"] = "unknown"
        out[f"food101_top{k}_conf"] = 0.0
    if cls_model is None:
        return out
    result = cls_model.predict(np.array(image.convert("RGB")), imgsz=YOLO_IMGSZ, verbose=False)[0]
    probs = getattr(result, "probs", None)
    if probs is None:
        return out
    top_ids = list(probs.top5[:topk])
    top_confs = [float(x) for x in probs.top5conf[:topk]]
    for i, (class_id, conf) in enumerate(zip(top_ids, top_confs), start=1):
        name = str(cls_model.names.get(int(class_id), class_id))
        out[f"food101_top{i}"] = name
        out[f"food101_top{i}_conf"] = conf
        if i == 1:
            out["food101_dish_group"] = food101_group_by_name.get(name, "unknown")
    conf_arr = np.asarray(top_confs, dtype=np.float32)
    conf_arr = conf_arr / max(float(conf_arr.sum()), 1e-6)
    out["food101_entropy"] = float(-(conf_arr * np.log(conf_arr + 1e-9)).sum())
    return out


In [6]:
def height_features(mask: np.ndarray, depth_norm: np.ndarray, prefix: str, ring_fracs=(0.015, 0.035, 0.07)) -> Dict[str, float]:
    h, w = mask.shape[:2]
    out = {}
    food_values = depth_norm[mask]
    food_values = food_values[np.isfinite(food_values)]
    if food_values.size == 0:
        for frac in ring_fracs:
            tag = f"{prefix}_ring{int(frac * 1000):03d}"
            out[f"{tag}_plate_depth"] = 0.0
            out[f"{tag}_vol_plate_minus_food"] = 0.0
            out[f"{tag}_vol_food_minus_plate"] = 0.0
            out[f"{tag}_mean_abs_height"] = 0.0
            out[f"{tag}_p75_abs_height"] = 0.0
            out[f"{tag}_p95_abs_height"] = 0.0
        return out

    for frac in ring_fracs:
        kernel_size = max(5, int(round(min(h, w) * frac)))
        if kernel_size % 2 == 0:
            kernel_size += 1
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        dilated = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
        ring = dilated & ~mask
        ring_values = depth_norm[ring]
        ring_values = ring_values[np.isfinite(ring_values)]
        if ring_values.size < 20:
            ring_values = depth_norm[~mask]
            ring_values = ring_values[np.isfinite(ring_values)]
        plate_depth = float(np.median(ring_values)) if ring_values.size else float(np.median(depth_norm[np.isfinite(depth_norm)]))
        plate_minus_food = np.clip(plate_depth - food_values, 0, None)
        food_minus_plate = np.clip(food_values - plate_depth, 0, None)
        if plate_minus_food.size:
            plate_minus_food = np.clip(plate_minus_food, 0, np.percentile(plate_minus_food, 95))
        if food_minus_plate.size:
            food_minus_plate = np.clip(food_minus_plate, 0, np.percentile(food_minus_plate, 95))
        tag = f"{prefix}_ring{int(frac * 1000):03d}"
        out[f"{tag}_plate_depth"] = plate_depth
        out[f"{tag}_vol_plate_minus_food"] = float(plate_minus_food.sum())
        out[f"{tag}_vol_food_minus_plate"] = float(food_minus_plate.sum())
        out[f"{tag}_mean_abs_height"] = float(np.mean(np.abs(food_values - plate_depth)))
        out[f"{tag}_p75_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 75))
        out[f"{tag}_p95_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 95))
    return out


def plate_relation_features(food_mask: np.ndarray, plate_mask: np.ndarray, image_area: int) -> Dict[str, float]:
    food_area = int(food_mask.sum())
    plate_area = int(plate_mask.sum())
    intersection = int((food_mask & plate_mask).sum()) if plate_area else 0
    return {
        "plate_area_px": plate_area,
        "plate_area_ratio": float(plate_area / max(image_area, 1)),
        "food_to_plate_area_ratio": float(food_area / max(plate_area, 1)) if plate_area else 0.0,
        "food_plate_intersection_px": intersection,
        "food_in_plate_ratio": float(intersection / max(food_area, 1)) if food_area else 0.0,
        "plate_covered_by_food_ratio": float(intersection / max(plate_area, 1)) if plate_area else 0.0,
    }


def compute_image_features(instances: List[Dict[str, object]], plate_mask: np.ndarray, depth: np.ndarray, image_size: Tuple[int, int]) -> Dict[str, object]:
    h, w = image_size
    depth = resize_depth(depth, (h, w))
    depth_versions = normalize_depth(depth)
    plate_mask = resize_nearest(plate_mask, (h, w))

    out: Dict[str, object] = {"image_h": h, "image_w": w, "image_area_px": h * w}
    for group in density_groups:
        out[f"area_group_{group}"] = 0.0
        out[f"count_group_{group}"] = 0.0
        out[f"pvol_group_{group}"] = 0.0

    if not instances:
        empty_food = np.zeros((h, w), dtype=bool)
        out.update({
            "area_px": 0,
            "sqrt_area": 0.0,
            "log_area": 0.0,
            "area_ratio": 0.0,
            "n_unique_seg_classes": 0,
            "dominant_density_group": "unknown",
            "dominant_seg_class": "unknown",
        })
        for rank in range(1, 4):
            out[f"top{rank}_seg_class"] = "unknown"
            out[f"top{rank}_density_group"] = "unknown"
            out[f"top{rank}_area_ratio"] = 0.0
        out.update(mask_shape_features(empty_food, "union"))
        out.update(mask_shape_features(plate_mask, "plate"))
        out.update(plate_relation_features(empty_food, plate_mask, h * w))
        return out

    union_mask = np.logical_or.reduce([inst["mask"] for inst in instances])
    area_px = int(union_mask.sum())
    out.update({
        "area_px": area_px,
        "sqrt_area": float(math.sqrt(area_px)),
        "log_area": float(math.log1p(area_px)),
        "area_ratio": float(area_px / max(h * w, 1)),
        "n_unique_seg_classes": int(len(set(inst["class_id"] for inst in instances))),
    })
    out.update(mask_shape_features(union_mask, "union"))
    out.update(mask_shape_features(plate_mask, "plate"))
    out.update(plate_relation_features(union_mask, plate_mask, h * w))

    instances_sorted = sorted(instances, key=lambda x: int(x["area"]), reverse=True)
    for rank in range(1, 4):
        if len(instances_sorted) >= rank:
            inst = instances_sorted[rank - 1]
            out[f"top{rank}_seg_class"] = inst["class_name"]
            out[f"top{rank}_density_group"] = inst["density_group"]
            out[f"top{rank}_area_ratio"] = float(inst["area"] / max(area_px, 1))
        else:
            out[f"top{rank}_seg_class"] = "unknown"
            out[f"top{rank}_density_group"] = "unknown"
            out[f"top{rank}_area_ratio"] = 0.0
    out["dominant_seg_class"] = out["top1_seg_class"]
    out["dominant_density_group"] = out["top1_density_group"]

    p05p95 = depth_versions["p05p95"]
    out.update(height_features(union_mask, p05p95, "union_p05p95"))
    out.update(height_features(union_mask, depth_versions["iqr_z"], "union_iqrz"))

    for inst in instances:
        group = str(inst["density_group"])
        mask = inst["mask"]
        out[f"area_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) + int(inst["area"]))
        out[f"count_group_{group}"] = float(out.get(f"count_group_{group}", 0.0) + 1.0)
        hf = height_features(mask, p05p95, "tmp")
        out[f"pvol_group_{group}"] = float(out.get(f"pvol_group_{group}", 0.0) + hf.get("tmp_ring035_vol_plate_minus_food", 0.0) + hf.get("tmp_ring035_vol_food_minus_plate", 0.0))

    for group in density_groups:
        out[f"area_ratio_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) / max(area_px, 1))

    return out


## 5. Feature extraction


In [7]:
seg_model = YOLO(str(SEG_MODEL_PATH))
plate_model = YOLO(str(PLATE_MODEL_PATH))
cls_model = YOLO(str(CLS_MODEL_PATH)) if CLS_MODEL_PATH else None
depth_engine = DepthAnything3Module(model_id=DEPTH_MODEL_ID)

feature_rows: List[Dict[str, object]] = []
done_ids = set()
if RESUME_FEATURE_CACHE and FEATURES_CSV.exists():
    cached = pd.read_csv(FEATURES_CSV)
    feature_rows = cached.to_dict("records")
    done_ids = set(cached["dish_id"].astype(str))
    print(f"Loaded cached feature rows: {len(done_ids):,}")

pending = labels[~labels["dish_id"].astype(str).isin(done_ids)].copy()
print(f"Pending dishes: {len(pending):,}")


Pending dishes: 3,244


In [8]:
errors = []
started_at = time.time()

for _, row in tqdm(list(pending.iterrows()), total=len(pending)):
    dish_id = str(row["dish_id"])
    image_path = Path(row["rgb_path"])
    try:
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        instances, seg_stats = predict_seg_instances(seg_model, image)
        plate_mask, plate_stats = predict_plate_mask(plate_model, image)
        depth = np.asarray(depth_engine.get_depth_matrix(str(image_path)), dtype=np.float32)
        feature_rows.append({
            "dish_id": dish_id,
            "image_path": str(image_path.relative_to(PROJECT_ROOT)),
            "mass": float(row["mass"]),
            "calories": float(row["calories"]),
            "fat": float(row["fat"]),
            "carb": float(row["carb"]),
            "protein": float(row["protein"]),
            **seg_stats,
            **plate_stats,
            **compute_image_features(instances, plate_mask, depth, (height, width)),
            **predict_food101(cls_model, image),
        })
    except Exception as exc:
        errors.append({"dish_id": dish_id, "error": repr(exc)})

    if len(feature_rows) % CACHE_EVERY_N_ROWS == 0:
        pd.DataFrame(feature_rows).to_csv(FEATURES_CSV, index=False)

features = pd.DataFrame(feature_rows)
features.to_csv(FEATURES_CSV, index=False)
print(f"Feature rows: {len(features):,}")
print(f"Errors: {len(errors):,}")
print(f"Elapsed minutes: {(time.time() - started_at) / 60:.1f}")
features.head()


100%|██████████| 3244/3244 [14:41<00:00,  3.68it/s]


Feature rows: 3,244
Errors: 0
Elapsed minutes: 14.7


,dish_id,image_path,mass,calories,fat,carb,protein,n_masks_raw,n_masks_kept,seg_conf_mean,...,food101_dish_group,food101_entropy,food101_top2,food101_top2_conf,food101_top3,food101_top3_conf,food101_top4,food101_top4_conf,food101_top5,food101_top5_conf
0,dish_1561662216,nutrition5k\imagery\realsense_overhead\dish_15...,193.0,300.794281,12.387489,28.218290,18.633970,19,19,0.390714,...,starch_main,1.540739,seaweed_salad,0.102099,ceviche,0.075203,guacamole,0.065337,sushi,0.056923
1,dish_1561662054,nutrition5k\imagery\realsense_overhead\dish_15...,292.0,419.438782,23.838249,26.351543,25.910593,18,18,0.490134,...,salad,1.544497,seaweed_salad,0.094088,risotto,0.085639,greek_salad,0.071085,ceviche,0.065655
2,dish_1562008979,nutrition5k\imagery\realsense_overhead\dish_15...,290.0,382.936646,22.224644,10.173570,35.345387,16,16,0.418948,...,starch_main,1.242722,seaweed_salad,0.060776,tacos,0.046388,guacamole,0.040875,falafel,0.037696
3,dish_1560455030,nutrition5k\imagery\realsense_overhead\dish_15...,103.0,20.590000,0.148000,4.625000,0.956000,12,11,0.363725,...,seafood,1.489761,tacos,0.092264,seaweed_salad,0.065725,frozen_yogurt,0.040688,sushi,0.034704
4,dish_1558372433,nutrition5k\imagery\realsense_overhead\dish_15...,143.0,74.360001,0.286000,0.429000,20.020000,8,8,0.492923,...,dairy_dessert,1.555149,lobster_bisque,0.073019,panna_cotta,0.063798,foie_gras,0.053099,ice_cream,0.049872


## 6. CatBoost training


In [9]:
features = pd.read_csv(FEATURES_CSV)
features = features.replace([np.inf, -np.inf], np.nan).dropna(subset=["mass"])

report_cols = ["calories", "fat", "carb", "protein"]
drop_cols = ["dish_id", "image_path", "mass", *report_cols]
feature_cols = [c for c in features.columns if c not in drop_cols]
cat_cols = [
    c for c in feature_cols
    if pd.api.types.is_object_dtype(features[c])
    or pd.api.types.is_string_dtype(features[c])
    or isinstance(features[c].dtype, pd.CategoricalDtype)
]
num_cols = [c for c in feature_cols if c not in cat_cols]

features[cat_cols] = features[cat_cols].fillna("unknown").astype(str)
features[num_cols] = features[num_cols].fillna(0.0)

train_df, valid_df = train_test_split(features, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_valid = train_df[feature_cols], valid_df[feature_cols]
y_train = np.log1p(train_df["mass"].astype(float))
y_valid_log = np.log1p(valid_df["mass"].astype(float))
y_valid = valid_df["mass"].astype(float)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool = Pool(X_valid, y_valid_log, cat_features=cat_cols)
print(f"Train rows: {len(train_df):,}; valid rows: {len(valid_df):,}")
print(f"Features: {len(feature_cols)}; categorical: {len(cat_cols)}")


Train rows: 2,595; valid rows: 649
Features: 185; categorical: 14


In [10]:
model = CatBoostRegressor(**CATBOOST_PARAMS)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

pred_mass = np.clip(np.expm1(model.predict(valid_pool)), 0, None)
valid_report = valid_df[["dish_id", "mass", "calories", "fat", "carb", "protein"]].copy()
valid_report["pred_mass_g"] = pred_mass
valid_report["abs_error_g"] = np.abs(valid_report["mass"] - valid_report["pred_mass_g"])
valid_report["mass_bin"] = pd.cut(valid_report["mass"], bins=[0, 100, 250, 500, np.inf], labels=["0-100", "100-250", "250-500", "500+"])

for col in ["calories", "fat", "carb", "protein"]:
    per_g = valid_report[col].astype(float) / valid_report["mass"].astype(float).clip(lower=1e-6)
    valid_report[f"{col}_per_100g"] = per_g * 100.0
    valid_report[f"pred_{col}"] = per_g * valid_report["pred_mass_g"].astype(float)

metrics = {
    "valid_mae_g": float(mean_absolute_error(y_valid, pred_mass)),
    "valid_rmse_g": float(math.sqrt(mean_squared_error(y_valid, pred_mass))),
    "valid_r2": float(r2_score(y_valid, pred_mass)),
    "baseline_median_mae_g": float(mean_absolute_error(y_valid, np.full_like(y_valid, train_df["mass"].median(), dtype=float))),
    "bin_mae_g": {str(k): float(v) for k, v in valid_report.groupby("mass_bin", observed=False)["abs_error_g"].mean().to_dict().items()},
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    "features_csv": str(FEATURES_CSV.relative_to(PROJECT_ROOT)),
    "valid_report_csv": str(VALID_REPORT_CSV.relative_to(PROJECT_ROOT)),
}

model.save_model(MODEL_PATH)
valid_report.to_csv(VALID_REPORT_CSV, index=False)
METRICS_PATH.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metrics, indent=2, ensure_ascii=False))
valid_report[[
    "dish_id", "mass", "pred_mass_g", "abs_error_g",
    "pred_calories", "pred_fat", "pred_carb", "pred_protein",
    "calories_per_100g", "fat_per_100g", "carb_per_100g", "protein_per_100g",
]].head(20)


0:	learn: 0.6613583	test: 0.6595235	best: 0.6595235 (0)	total: 203ms	remaining: 10m 8s
100:	learn: 0.3557157	test: 0.4259063	best: 0.4259063 (100)	total: 6.61s	remaining: 3m 9s
200:	learn: 0.3002628	test: 0.4164952	best: 0.4163991 (198)	total: 12.6s	remaining: 2m 55s
300:	learn: 0.2590762	test: 0.4137612	best: 0.4137612 (300)	total: 18.8s	remaining: 2m 48s
400:	learn: 0.2240957	test: 0.4121531	best: 0.4121531 (400)	total: 24.7s	remaining: 2m 40s
500:	learn: 0.2000638	test: 0.4115737	best: 0.4115737 (500)	total: 30.9s	remaining: 2m 33s
600:	learn: 0.1798936	test: 0.4114494	best: 0.4108804 (553)	total: 37.2s	remaining: 2m 28s
700:	learn: 0.1631103	test: 0.4118305	best: 0.4108804 (553)	total: 43.2s	remaining: 2m 21s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.4108803808
bestIteration = 553

Shrink model to first 554 iterations.
{
  "valid_mae_g": 78.62302094012844,
  "valid_rmse_g": 112.26686050134492,
  "valid_r2": 0.49186088785767357,
  "baseline_median_mae_g": 

,dish_id,mass,pred_mass_g,abs_error_g,pred_calories,pred_fat,pred_carb,pred_protein,calories_per_100g,fat_per_100g,carb_per_100g,protein_per_100g
2585,dish_1565118782,353.0,210.445298,142.554702,332.373373,20.334560,10.162539,26.076711,157.938132,9.662635,4.829065,12.391207
3120,dish_1561576794,738.0,422.667992,315.332008,374.972516,16.122907,21.586914,36.343666,88.715617,3.814556,5.107298,8.598632
1124,dish_1561664042,209.0,317.446151,108.446151,432.946426,14.796360,14.448873,57.460665,136.384210,4.661061,4.551598,18.100917
321,dish_1558635994,136.0,79.297285,56.702715,54.715124,0.158595,14.273512,0.555081,68.999997,0.200000,18.000001,0.700000
2844,dish_1563898084,384.0,218.980869,165.019131,239.221418,13.023361,17.108596,14.240557,109.243067,5.947260,7.812827,6.503106
1080,dish_1568665052,331.0,391.326449,60.326449,408.651279,19.106468,20.465235,39.427097,104.427206,4.882488,5.229709,10.075245
2452,dish_1568060414,146.0,125.484107,20.515893,46.066898,1.833826,6.833369,2.262870,36.711340,1.461401,5.445605,1.803312
1211,dish_1563823292,208.0,136.670509,71.329491,296.040625,19.293014,5.060638,26.143518,216.609001,14.116442,3.702801,19.128865
2276,dish_1566332154,425.0,360.827264,64.172736,340.493577,14.979851,9.641305,42.671858,94.364703,4.151530,2.672000,11.826118
1003,dish_1565897617,211.0,187.984473,23.015527,317.909172,20.276611,8.657489,26.517258,169.114591,10.786322,4.605427,14.106090
